In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.order_events AS
WITH dedup AS (
  -- keep the first-ingested copy of every event_id
  SELECT event_id, order_id, sub_order_id, event_type, event_ts, sku_id, rider_id, cancel_reason, ingested_at
  FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY event_id ORDER BY ingested_at) AS rn
    FROM workspace.bronze.order_events
  )
  WHERE rn = 1
),
final_rider AS (
  -- the rider who finished a shipment = rider on its latest event that still has a rider_id
  SELECT sub_order_id, max_by(rider_id, event_ts) AS final_rider_id
  FROM dedup
  WHERE rider_id IS NOT NULL
  GROUP BY sub_order_id
),
last_assign AS (
  SELECT sub_order_id, MAX(event_ts) AS last_assign_ts
  FROM dedup
  WHERE event_type = 'rider_assigned'
  GROUP BY sub_order_id
),
filled AS (
  SELECT d.*,
         CASE
           WHEN d.rider_id IS NOT NULL THEN d.rider_id
           WHEN d.event_type IN ('rider_arrived_at_store', 'order_picked_up', 'order_delivered')
                THEN fr.final_rider_id
           WHEN d.event_type = 'rider_assigned' AND d.event_ts = la.last_assign_ts
                THEN fr.final_rider_id
         END AS rider_id_clean
  FROM dedup d
  LEFT JOIN final_rider fr ON d.sub_order_id = fr.sub_order_id
  LEFT JOIN last_assign la ON d.sub_order_id = la.sub_order_id
)
SELECT event_id, order_id, sub_order_id, event_type, event_ts, sku_id,
       rider_id_clean AS rider_id,
       (rider_id IS NULL AND rider_id_clean IS NOT NULL) AS rider_id_imputed,
       cancel_reason, ingested_at,
       (unix_timestamp(ingested_at) - unix_timestamp(event_ts) > 1800) AS is_late_arrival
FROM filled;

In [0]:
%sql
SELECT COUNT(*) AS rows_total,
       COUNT(DISTINCT event_id) AS distinct_events,
       SUM(CASE WHEN rider_id_imputed THEN 1 ELSE 0 END) AS rider_ids_filled,
       SUM(CASE WHEN is_late_arrival THEN 1 ELSE 0 END) AS late_rows
FROM workspace.silver.order_events;

In [0]:
%sql
--sub order timeline

CREATE OR REPLACE TABLE workspace.silver.sub_order_timeline AS
SELECT s.sub_order_id, s.order_id, s.store_id, s.sub_order_seq,
       MIN(CASE WHEN e.event_type = 'picking_started'        THEN e.event_ts END) AS picking_started_ts,
       MIN(CASE WHEN e.event_type = 'packing_completed'      THEN e.event_ts END) AS packing_completed_ts,
       COUNT(CASE WHEN e.event_type = 'rider_assigned'       THEN 1 END)          AS n_rider_assignments,
       MAX(CASE WHEN e.event_type = 'rider_assigned'         THEN e.event_ts END) AS rider_assigned_ts,
       MIN(CASE WHEN e.event_type = 'rider_arrived_at_store' THEN e.event_ts END) AS rider_arrived_ts,
       MIN(CASE WHEN e.event_type = 'order_picked_up'        THEN e.event_ts END) AS picked_up_ts,
       MIN(CASE WHEN e.event_type = 'order_delivered'        THEN e.event_ts END) AS delivered_ts,
       MAX(CASE WHEN e.event_type = 'order_delivered'        THEN e.rider_id END) AS rider_id
FROM workspace.bronze.sub_orders s
LEFT JOIN workspace.silver.order_events e ON e.sub_order_id = s.sub_order_id
GROUP BY s.sub_order_id, s.order_id, s.store_id, s.sub_order_seq;

In [0]:
%sql
SELECT COUNT(*) AS shipments,
       COUNT(delivered_ts) AS delivered_shipments,
       SUM(CASE WHEN delivered_ts IS NULL THEN 1 ELSE 0 END) AS not_delivered,
       SUM(CASE WHEN sub_order_seq = 2 THEN 1 ELSE 0 END) AS second_shipments,
       SUM(CASE WHEN n_rider_assignments > 1 THEN 1 ELSE 0 END) AS reassigned
FROM workspace.silver.sub_order_timeline;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.orders AS
WITH oe AS (
  SELECT order_id,
         MIN(CASE WHEN event_type = 'order_accepted'  THEN event_ts END)      AS accepted_ts,
         MIN(CASE WHEN event_type = 'order_cancelled' THEN event_ts END)      AS cancelled_ts,
         MAX(CASE WHEN event_type = 'order_cancelled' THEN cancel_reason END) AS cancel_reason
  FROM workspace.silver.order_events
  GROUP BY order_id
),
st AS (
  SELECT order_id,
         COUNT(*)          AS n_sub_orders,
         COUNT(delivered_ts) AS n_delivered,
         MAX(delivered_ts) AS last_delivered_ts      -- an order is done when its LAST shipment arrives
  FROM workspace.silver.sub_order_timeline
  GROUP BY order_id
)
SELECT o.order_id, o.customer_id, o.store_id, o.placed_ts,
       CAST(o.placed_ts AS DATE) AS placed_date,
       HOUR(o.placed_ts)         AS placed_hour,
       o.promised_minutes,
       oe.accepted_ts, oe.cancelled_ts, oe.cancel_reason,
       st.n_sub_orders,
       (st.n_sub_orders > 1) AS is_split,
       CASE WHEN oe.cancelled_ts IS NOT NULL THEN 'cancelled'
            WHEN st.n_delivered = st.n_sub_orders THEN 'delivered'
            ELSE 'incomplete' END AS order_status,
       CASE WHEN oe.cancelled_ts IS NULL AND st.n_delivered = st.n_sub_orders
            THEN st.last_delivered_ts END AS delivered_ts
FROM workspace.bronze.orders o
LEFT JOIN oe ON o.order_id = oe.order_id
LEFT JOIN st ON o.order_id = st.order_id;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.order_items AS
WITH unav AS (
  SELECT DISTINCT order_id, sku_id, TRUE AS was_unavailable
  FROM workspace.silver.order_events
  WHERE event_type = 'item_unavailable'
)
SELECT i.order_id, i.sku_id, i.sub_order_id, i.qty_ordered, i.unit_price, i.qty_fulfilled,
       i.item_status, i.substitute_sku_id,
       i.qty_ordered * i.unit_price AS ordered_value,
       COALESCE(u.was_unavailable, FALSE) AS was_unavailable
FROM workspace.bronze.order_items i
LEFT JOIN unav u ON i.order_id = u.order_id AND i.sku_id = u.sku_id;